In [1]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


Saving 100_company_main.csv to 100_company_main.csv
User uploaded file "100_company_main.csv" with length 19586 bytes


In [6]:
import pandas as pd

company_100_main_df = pd.read_csv("100_company_main.csv")
company_100_main_df.head()

,Ticker,Total_ESG,Environmental,Social,Governance,Sum_sub,Diff,Anomaly,Country,Sector,Industry,MarketCap,TotalRevenue,NetIncome,ReturnOnEquity,DebtToEquity,CurrentRatio,QuickRatio,Region,Yahoo_Finance_Name
0,LIN,11.63,8.23,1.31,2.09,11.63,0.00,False,United Kingdom,Basic Materials,Specialty Chemicals,211426623488,33244999680,6713999872,0.17313,64.823,0.926,0.698,Europe,Linde plc
1,ALV,15.24,6.26,4.88,4.10,15.24,0.00,False,Sweden,Consumer Cyclical,Auto Parts,8961631232,10613999616,752000000,0.31007,85.580,0.953,0.624,Europe,"Autoliv, Inc."
2,ACN,12.88,2.54,6.35,3.99,12.88,0.00,False,Ireland,Technology,Information Technology Services,147834912768,69672976384,7678432768,0.25509,25.380,1.420,1.301,Europe,Accenture plc
3,ETN,19.80,7.44,8.57,3.80,19.81,-0.01,False,Ireland,Industrials,Specialty Industrial Machinery,145325686784,25988999168,3924999936,0.20733,62.391,1.240,0.686,Europe,Eaton Corporation plc
4,MDT,16.27,2.35,9.01,4.91,16.27,0.00,False,Ireland,Healthcare,Medical Devices,122855645184,34200000512,4659999744,0.09741,59.437,2.014,1.248,Europe,Medtronic plc


In [8]:
import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output


# ========= DATAFRAME =========
# Must exist from previous cell:
# company_100_main_df


# ========= COLUMN MAP (EXACTLY YOUR STRUCTURE) =========
COLS = {
    "company": "Ticker",
    "sector": "Sector",
    "total_esg": "Total_ESG",
    "E": "Environmental",
    "S": "Social",
    "G": "Governance",
    "roe": "ReturnOnEquity",
    "de": "DebtToEquity",
}

# ========= ESG RISK CATEGORIES (Sustainalytics) =========
ESG_MAX_BY_CATEGORY = {
    "Nenozīmīgs (≤10)": 10.0,
    "Zems (≤20)": 20.0,
    "Vidējs (≤30)": 30.0,
    "Augsts (≤40)": 40.0,
    "Ļoti augsts (>40)": np.inf,
}

# ========= FINANCIAL RISK PROFILES (FROM YOUR SAMPLE QUANTILES) =========
FIN_PROFILES = {
    "Konservatīvs": {"max_de": 66.071,    "min_roe": 0.146505},
    "Sabalansēts":  {"max_de": 142.39775, "min_roe": 0.073203},
    "Agresīvs":     {"max_de": 142.39775, "min_roe": 0.0},
}

# Scoring-only cap for D/E to reduce outlier dominance (does NOT affect filtering)
DE_WINSOR_PCT = 0.95


def _require_columns(df: pd.DataFrame):
    missing = [v for v in COLS.values() if v not in df.columns]
    if missing:
        raise KeyError(f"Trūkst kolonnas: {missing}\nPieejamās kolonnas: {list(df.columns)}")


def _normalize_weights(wE, wS, wG):
    w = np.array([wE, wS, wG], dtype=float)
    s = w.sum()
    if s <= 0:
        return 1/3, 1/3, 1/3
    w = w / s
    return float(w[0]), float(w[1]), float(w[2])


def _minmax_0_1(series: pd.Series, higher_is_better: bool = True):
    x = series.astype(float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    if np.isclose(mx, mn):
        z = np.ones(len(x)) * 0.5
        return pd.Series(z, index=series.index)
    z = (x - mn) / (mx - mn)
    return z if higher_is_better else (1 - z)


def build_portfolio(
    df: pd.DataFrame,
    esg_category: str,
    fin_profile: str,
    wE: float, wS: float, wG: float,
    alpha_esg_vs_fin: float,     # 0..1
    beta_roe_vs_de: float,       # 0..1
    n_total: int,
    n_sectors: int,
    max_per_sector: int,
):
    _require_columns(df)

    esg_max = ESG_MAX_BY_CATEGORY[esg_category]
    prof = FIN_PROFILES[fin_profile]
    max_de = prof["max_de"]
    min_roe = prof["min_roe"]

    wE, wS, wG = _normalize_weights(wE, wS, wG)

    d = df.copy()

    # Numeric conversion
    for k in ["total_esg", "E", "S", "G", "roe", "de"]:
        d[COLS[k]] = pd.to_numeric(d[COLS[k]], errors="coerce")

    # Drop rows with missing key fields
    key_fields = [
        COLS["company"], COLS["sector"],
        COLS["total_esg"], COLS["E"], COLS["S"], COLS["G"],
        COLS["roe"], COLS["de"]
    ]
    d = d.dropna(subset=key_fields).copy()

    # === FILTERS (risk tolerance) ===
    # Total_ESG is treated as ESG RISK: lower is better
    d_f = d[
        (d[COLS["total_esg"]] <= esg_max) &
        (d[COLS["de"]] <= max_de) &
        (d[COLS["roe"]] >= min_roe)
    ].copy()

    meta = {
        "filters": {"esg_category": esg_category, "esg_max": esg_max, "fin_profile": fin_profile, "max_de": max_de, "min_roe": min_roe},
        "counts": {"initial": len(d), "after_filters": len(d_f)},
        "weights": {"wE": wE, "wS": wS, "wG": wG, "alpha": alpha_esg_vs_fin, "beta": beta_roe_vs_de},
        "diversification": {"n_total": n_total, "n_sectors_target": n_sectors, "max_per_sector": max_per_sector},
    }

    if d_f.empty:
        return None, {**meta, "message": "Neviens uzņēmums neizgāja cauri filtriem. Samazini stingrību (ESG/finanšu risku) vai palielini sliekšņus."}

    # === SCORES ===
    # ESG weighted risk -> convert to goodness score
    esg_weighted_risk = (wE * d_f[COLS["E"]] + wS * d_f[COLS["S"]] + wG * d_f[COLS["G"]])
    esg_score = _minmax_0_1(esg_weighted_risk, higher_is_better=False)

    # Financial block: ROE (higher better) and D/E (lower better)
    de_cap = d_f[COLS["de"]].quantile(DE_WINSOR_PCT)
    de_for_score = d_f[COLS["de"]].clip(upper=de_cap)

    roe_score = _minmax_0_1(d_f[COLS["roe"]], higher_is_better=True)
    de_score = _minmax_0_1(de_for_score, higher_is_better=False)

    fin_score = beta_roe_vs_de * roe_score + (1 - beta_roe_vs_de) * de_score

    final_score = alpha_esg_vs_fin * esg_score + (1 - alpha_esg_vs_fin) * fin_score

    d_f = d_f.assign(
        ESG_weighted_risk=esg_weighted_risk,
        ESG_score=esg_score,
        ROE_score=roe_score,
        DE_score=de_score,
        Financial_score=fin_score,
        Final_score=final_score,
    )

    # === DIVERSIFIED SELECTION ===
    ranked = d_f.sort_values("Final_score", ascending=False).copy()

    # Choose sectors by best score per sector
    sector_best = ranked.groupby(COLS["sector"], as_index=False).head(1).sort_values("Final_score", ascending=False)
    chosen_sectors = sector_best[COLS["sector"]].head(min(n_sectors, sector_best[COLS["sector"]].nunique())).tolist()

    # Seed portfolio with best from chosen sectors
    seed = ranked[ranked[COLS["sector"]].isin(chosen_sectors)].groupby(COLS["sector"], as_index=False).head(1)
    portfolio = seed.copy()

    sector_counts = portfolio[COLS["sector"]].value_counts().to_dict()
    used_idx = set(portfolio.index.tolist())

    for idx, row in ranked.iterrows():
        if len(portfolio) >= n_total:
            break
        if idx in used_idx:
            continue
        sec = row[COLS["sector"]]
        if sector_counts.get(sec, 0) >= max_per_sector:
            continue
        portfolio = pd.concat([portfolio, row.to_frame().T], axis=0)
        used_idx.add(idx)
        sector_counts[sec] = sector_counts.get(sec, 0) + 1

    # Fill if short (relax sector cap)
    if len(portfolio) < n_total:
        for idx, row in ranked.iterrows():
            if len(portfolio) >= n_total:
                break
            if idx in used_idx:
                continue
            portfolio = pd.concat([portfolio, row.to_frame().T], axis=0)
            used_idx.add(idx)

    portfolio = portfolio.sort_values("Final_score", ascending=False).copy()
    portfolio["Weight"] = 1.0 / len(portfolio)

    meta["counts"]["portfolio_size"] = len(portfolio)
    meta["diversification"]["n_sectors_achieved"] = int(portfolio[COLS["sector"]].nunique())

    out_cols = [
        COLS["company"], COLS["sector"],
        COLS["total_esg"], COLS["E"], COLS["S"], COLS["G"],
        COLS["roe"], COLS["de"],
        "ESG_weighted_risk", "ESG_score",
        "Financial_score", "Final_score",
        "Weight"
    ]
    return portfolio[out_cols].reset_index(drop=True), meta


# ========= GUI =========
esg_cat_dd = widgets.Dropdown(
    options=list(ESG_MAX_BY_CATEGORY.keys()),
    value="Zems (≤20)",
    description="ESG max:",
    layout=widgets.Layout(width="320px")
)

fin_prof_dd = widgets.Dropdown(
    options=list(FIN_PROFILES.keys()),
    value="Sabalansēts",
    description="Fin risks:",
    layout=widgets.Layout(width="320px")
)

wE_slider = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="wE", readout_format=".2f")
wS_slider = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="wS", readout_format=".2f")
wG_slider = widgets.FloatSlider(value=0.34, min=0.0, max=1.0, step=0.01, description="wG", readout_format=".2f")

alpha_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="α ESG↔Fin", readout_format=".2f")
beta_slider  = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="β ROE↔D/E", readout_format=".2f")

n_total_int = widgets.BoundedIntText(value=12, min=3, max=50, step=1, description="N")
n_sectors_int = widgets.BoundedIntText(value=6, min=1, max=20, step=1, description="Sektori")
max_per_sector_int = widgets.BoundedIntText(value=3, min=1, max=20, step=1, description="Max/sek")

run_btn = widgets.Button(description="Build portfolio", button_style="primary", icon="check")
out = widgets.Output()

def on_run_clicked(_):
    with out:
        clear_output()
        df = company_100_main_df

        port, meta = build_portfolio(
            df=df,
            esg_category=esg_cat_dd.value,
            fin_profile=fin_prof_dd.value,
            wE=wE_slider.value, wS=wS_slider.value, wG=wG_slider.value,
            alpha_esg_vs_fin=alpha_slider.value,
            beta_roe_vs_de=beta_slider.value,
            n_total=int(n_total_int.value),
            n_sectors=int(n_sectors_int.value),
            max_per_sector=int(max_per_sector_int.value),
        )

        wE, wS, wG = _normalize_weights(wE_slider.value, wS_slider.value, wG_slider.value)
        print(f"Normalizēti E/S/G svari: wE={wE:.3f}, wS={wS:.3f}, wG={wG:.3f}")
        print(f"Filtri: Total_ESG ≤ {meta['filters']['esg_max']} ({meta['filters']['esg_category']}), "
              f"DebtToEquity ≤ {meta['filters']['max_de']} ({meta['filters']['fin_profile']}), "
              f"ReturnOnEquity ≥ {meta['filters']['min_roe']}")
        print(f"Skaits: sākumā={meta['counts']['initial']}, pēc filtriem={meta['counts']['after_filters']}")
        print(f"Portfelis: N={meta['counts'].get('portfolio_size', 0)}, "
              f"sektori mērķis={meta['diversification']['n_sectors_target']}, "
              f"sektori sasniegts={meta['diversification'].get('n_sectors_achieved', 0)}, "
              f"max/sek={meta['diversification']['max_per_sector']}")
        print("-" * 90)

        if port is None:
            print(meta["message"])
        else:
            display(port)

run_btn.on_click(on_run_clicked)

ui = widgets.VBox([
    widgets.HBox([esg_cat_dd, fin_prof_dd]),
    widgets.HBox([wE_slider, wS_slider, wG_slider]),
    widgets.HBox([alpha_slider, beta_slider]),
    widgets.HBox([n_total_int, n_sectors_int, max_per_sector_int]),
    run_btn,
    out
])

display(ui)

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- Pretty labels ----------
def pct_label(x):
    return f"{int(round(x*100))}%"

# Widgets: Filters
esg_cat_dd = widgets.Dropdown(
    options=list(ESG_MAX_BY_CATEGORY.keys()),
    value="Zems (≤20)",
    description="ESG risks:",
    layout=widgets.Layout(width="420px")
)

fin_prof_dd = widgets.Dropdown(
    options=list(FIN_PROFILES.keys()),
    value="Sabalansēts",
    description="Fin. risks:",
    layout=widgets.Layout(width="420px")
)

# Widgets: Priorities (use plain language, not α/β)
alpha_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="Ilgtspēja:",
    readout_format=".2f",
    layout=widgets.Layout(width="520px"),
)
alpha_help = widgets.HTML(
    "<div style='color:#555; font-size:12px;'>"
    "0 = tikai finanšu kvalitāte, 1 = tikai ilgtspēja. Vidus = līdzsvars.</div>"
)

beta_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="Atdeve:",
    readout_format=".2f",
    layout=widgets.Layout(width="520px"),
)
beta_help = widgets.HTML(
    "<div style='color:#555; font-size:12px;'>"
    "0 = uzsvars uz stabilitāti (zemāks D/E), 1 = uzsvars uz atdevi (augstāks ROE).</div>"
)

# Widgets: ESG weights (raw), then we show normalized in summary
wE_slider = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="E prioritāte:", readout_format=".2f")
wS_slider = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="S prioritāte:", readout_format=".2f")
wG_slider = widgets.FloatSlider(value=0.34, min=0.0, max=1.0, step=0.01, description="G prioritāte:", readout_format=".2f")

# Portfolio structure
n_total_int = widgets.BoundedIntText(value=12, min=3, max=50, step=1, description="N (uzņēmumi):")
n_sectors_int = widgets.BoundedIntText(value=6, min=1, max=20, step=1, description="K (sektori):")
max_per_sector_int = widgets.BoundedIntText(value=3, min=1, max=20, step=1, description="Max / sektors:")

run_btn = widgets.Button(description="Build portfolio", button_style="primary", icon="check")

summary_out = widgets.Output()
result_out = widgets.Output()

def render_summary():
    with summary_out:
        clear_output()

        # compute normalized E/S/G weights
        wE_n, wS_n, wG_n = _normalize_weights(wE_slider.value, wS_slider.value, wG_slider.value)

        # resolve thresholds
        esg_max = ESG_MAX_BY_CATEGORY[esg_cat_dd.value]
        prof = FIN_PROFILES[fin_prof_dd.value]
        max_de = prof["max_de"]
        min_roe = prof["min_roe"]

        # quick count after filters
        d = company_100_main_df.copy()
        for k in ["total_esg","E","S","G","roe","de"]:
            d[COLS[k]] = pd.to_numeric(d[COLS[k]], errors="coerce")
        d = d.dropna(subset=[COLS["total_esg"], COLS["roe"], COLS["de"], COLS["sector"], COLS["company"]])

        after_filters = d[
            (d[COLS["total_esg"]] <= esg_max) &
            (d[COLS["de"]] <= max_de) &
            (d[COLS["roe"]] >= min_roe)
        ]

        # show readable summary
        display(widgets.HTML(
            "<div style='padding:10px; border:1px solid #ddd; border-radius:10px;'>"
            "<div style='font-weight:600; margin-bottom:6px;'>Konfigurācijas kopsavilkums</div>"
            f"<div><b>Filtri:</b> Total_ESG ≤ {esg_max} | DebtToEquity ≤ {max_de} | ReturnOnEquity ≥ {min_roe}</div>"
            f"<div><b>E/S/G svari (normalizēti):</b> E={wE_n:.2f}, S={wS_n:.2f}, G={wG_n:.2f}</div>"
            f"<div><b>Līdzsvars:</b> Ilgtspēja={pct_label(alpha_slider.value)} | Finanses={pct_label(1-alpha_slider.value)} "
            f"| Atdeve={pct_label(beta_slider.value)} | Stabilitāte={pct_label(1-beta_slider.value)}</div>"
            f"<div><b>Pēc filtriem paliek:</b> {len(after_filters)} uzņēmumi (no {len(d)})</div>"
            "</div>"
        ))

def on_any_change(change):
    render_summary()

# Update summary dynamically
for w in [esg_cat_dd, fin_prof_dd, wE_slider, wS_slider, wG_slider, alpha_slider, beta_slider]:
    w.observe(on_any_change, names="value")

def on_run_clicked(_):
    with result_out:
        clear_output()

        port, meta = build_portfolio(
            df=company_100_main_df,
            esg_category=esg_cat_dd.value,
            fin_profile=fin_prof_dd.value,
            wE=wE_slider.value, wS=wS_slider.value, wG=wG_slider.value,
            alpha_esg_vs_fin=alpha_slider.value,
            beta_roe_vs_de=beta_slider.value,
            n_total=int(n_total_int.value),
            n_sectors=int(n_sectors_int.value),
            max_per_sector=int(max_per_sector_int.value),
        )

        if port is None:
            print(meta["message"])
            return

        display(port)

run_btn.on_click(on_run_clicked)

# ---------- Layout ----------
filters_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>1) Riska tolerance (filtri)</h3>"),
    widgets.HBox([esg_cat_dd, fin_prof_dd]),
])

priorities_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>2) Prioritātes (līdzsvars)</h3>"),
    alpha_slider, alpha_help,
    beta_slider, beta_help,
])

esg_weights_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>3) ESG fokuss (E/S/G)</h3>"),
    wE_slider, wS_slider, wG_slider,
    widgets.HTML("<div style='color:#555; font-size:12px;'>Svari tiek automātiski normalizēti, lai summa būtu 1.</div>")
])

portfolio_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>4) Portfeļa uzbūve</h3>"),
    widgets.HBox([n_total_int, n_sectors_int, max_per_sector_int]),
])

ui = widgets.VBox([
    filters_box,
    priorities_box,
    esg_weights_box,
    portfolio_box,
    summary_out,
    run_btn,
    result_out
])

render_summary()
display(ui)

In [10]:
# =========================================================
# ESG–FINANCE PORTFOLIO PROTOTYPE (Colab GUI, full code)
# File: 100_company_main.csv
# =========================================================

import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ----------------------------
# 1) LOAD DATA
# ----------------------------
company_100_main_df = pd.read_csv("100_company_main.csv")

# ----------------------------
# 2) COLUMN MAP (your exact structure)
# ----------------------------
COLS = {
    "company": "Ticker",
    "name": "Yahoo_Finance_Name",
    "sector": "Sector",
    "total_esg": "Total_ESG",
    "E": "Environmental",
    "S": "Social",
    "G": "Governance",
    "roe": "ReturnOnEquity",
    "de": "DebtToEquity",
}

# ----------------------------
# 3) ESG risk categories (Sustainalytics-style)
# ----------------------------
ESG_MAX_BY_CATEGORY = {
    "Nenozīmīgs (≤10)": 10.0,
    "Zems (≤20)": 20.0,
    "Vidējs (≤30)": 30.0,
    "Augsts (≤40)": 40.0,
    "Ļoti augsts (>40)": np.inf,
}

# ----------------------------
# 4) Financial risk profiles (based on your sample quantiles)
# ----------------------------
FIN_PROFILES = {
    "Konservatīvs": {"max_de": 66.071,    "min_roe": 0.146505},
    "Sabalansēts":  {"max_de": 142.39775, "min_roe": 0.073203},
    "Agresīvs":     {"max_de": 142.39775, "min_roe": 0.0},
}

# Scoring-only cap for D/E to reduce outlier dominance (does NOT affect filtering)
DE_WINSOR_PCT = 0.95


# ----------------------------
# 5) UTILITIES
# ----------------------------
def _require_columns(df: pd.DataFrame):
    missing = [v for v in COLS.values() if v not in df.columns]
    if missing:
        raise KeyError(f"Trūkst kolonnas: {missing}\nPieejamās kolonnas: {list(df.columns)}")

def _normalize_weights(wE, wS, wG):
    w = np.array([wE, wS, wG], dtype=float)
    s = w.sum()
    if s <= 0:
        return 1/3, 1/3, 1/3
    w = w / s
    return float(w[0]), float(w[1]), float(w[2])

def _minmax_0_1(series: pd.Series, higher_is_better: bool = True):
    x = pd.to_numeric(series, errors="coerce").astype(float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    if np.isclose(mx, mn):
        return pd.Series(np.ones(len(x)) * 0.5, index=series.index)
    z = (x - mn) / (mx - mn)
    return z if higher_is_better else (1 - z)

def pct_label(x):
    return f"{int(round(x*100))}%"

def plot_portfolio_charts(port: pd.DataFrame):
    # A) Sector distribution
    sec_counts = port[COLS["sector"]].value_counts().sort_values(ascending=False)
    plt.figure()
    sec_counts.plot(kind="bar")
    plt.title("Sector distribution in portfolio")
    plt.xlabel("Sector")
    plt.ylabel("Number of companies")
    plt.xticks(rotation=45, ha="right")
    plt.show()

    # B) ESG vs Financial scatter (with ticker labels)
    plt.figure()
    plt.scatter(port["Financial_score"], port["ESG_score"])
    for _, row in port.iterrows():
        plt.annotate(row[COLS["company"]], (row["Financial_score"], row["ESG_score"]), fontsize=8)
    plt.title("ESG score vs Financial score (portfolio)")
    plt.xlabel("Financial_score")
    plt.ylabel("ESG_score")
    plt.show()

    # C) Final score bar chart
    plt.figure()
    plt.bar(port[COLS["company"]], port["Final_score"])
    plt.title("Final_score of selected companies")
    plt.xlabel("Ticker")
    plt.ylabel("Final_score")
    plt.xticks(rotation=45, ha="right")
    plt.show()


# ----------------------------
# 6) CORE: build portfolio
# ----------------------------
def build_portfolio(
    df: pd.DataFrame,
    esg_category: str,
    fin_profile: str,
    wE: float, wS: float, wG: float,
    alpha_esg_vs_fin: float,     # 0..1
    beta_roe_vs_de: float,       # 0..1
    n_total: int,
    n_sectors: int,
    max_per_sector: int,
):
    _require_columns(df)

    esg_max = ESG_MAX_BY_CATEGORY[esg_category]
    prof = FIN_PROFILES[fin_profile]
    max_de = prof["max_de"]
    min_roe = prof["min_roe"]

    wE, wS, wG = _normalize_weights(wE, wS, wG)

    d = df.copy()

    # Numeric conversion
    for k in ["total_esg", "E", "S", "G", "roe", "de"]:
        d[COLS[k]] = pd.to_numeric(d[COLS[k]], errors="coerce")

    # Drop rows with missing key fields
    key_fields = [
        COLS["company"], COLS["name"], COLS["sector"],
        COLS["total_esg"], COLS["E"], COLS["S"], COLS["G"],
        COLS["roe"], COLS["de"]
    ]
    d = d.dropna(subset=key_fields).copy()

    # === FILTERS (risk tolerance) ===
    # Total_ESG is treated as ESG RISK: lower is better
    d_f = d[
        (d[COLS["total_esg"]] <= esg_max) &
        (d[COLS["de"]] <= max_de) &
        (d[COLS["roe"]] >= min_roe)
    ].copy()

    meta = {
        "filters": {"esg_category": esg_category, "esg_max": esg_max, "fin_profile": fin_profile, "max_de": max_de, "min_roe": min_roe},
        "counts": {"initial": len(d), "after_filters": len(d_f)},
        "weights": {"wE": wE, "wS": wS, "wG": wG, "alpha": alpha_esg_vs_fin, "beta": beta_roe_vs_de},
        "diversification": {"n_total": n_total, "n_sectors_target": n_sectors, "max_per_sector": max_per_sector},
    }

    if d_f.empty:
        return None, {**meta, "message": "Neviens uzņēmums neizgāja cauri filtriem. Samazini stingrību (ESG/finanšu risku) vai palielini sliekšņus."}

    # === SCORES ===
    # ESG weighted risk -> convert to goodness score
    esg_weighted_risk = (wE * d_f[COLS["E"]] + wS * d_f[COLS["S"]] + wG * d_f[COLS["G"]])
    esg_score = _minmax_0_1(esg_weighted_risk, higher_is_better=False)

    # Financial block: ROE (higher better) and D/E (lower better)
    de_cap = d_f[COLS["de"]].quantile(DE_WINSOR_PCT)
    de_for_score = d_f[COLS["de"]].clip(upper=de_cap)

    roe_score = _minmax_0_1(d_f[COLS["roe"]], higher_is_better=True)
    de_score = _minmax_0_1(de_for_score, higher_is_better=False)

    fin_score = beta_roe_vs_de * roe_score + (1 - beta_roe_vs_de) * de_score
    final_score = alpha_esg_vs_fin * esg_score + (1 - alpha_esg_vs_fin) * fin_score

    d_f = d_f.assign(
        ESG_weighted_risk=esg_weighted_risk,
        ESG_score=esg_score,
        ROE_score=roe_score,
        DE_score=de_score,
        Financial_score=fin_score,
        Final_score=final_score,
    )

    # === DIVERSIFIED SELECTION ===
    ranked = d_f.sort_values("Final_score", ascending=False).copy()

    sector_best = ranked.groupby(COLS["sector"], as_index=False).head(1).sort_values("Final_score", ascending=False)
    chosen_sectors = sector_best[COLS["sector"]].head(min(n_sectors, sector_best[COLS["sector"]].nunique())).tolist()

    seed = ranked[ranked[COLS["sector"]].isin(chosen_sectors)].groupby(COLS["sector"], as_index=False).head(1)
    portfolio = seed.copy()

    sector_counts = portfolio[COLS["sector"]].value_counts().to_dict()
    used_idx = set(portfolio.index.tolist())

    for idx, row in ranked.iterrows():
        if len(portfolio) >= n_total:
            break
        if idx in used_idx:
            continue
        sec = row[COLS["sector"]]
        if sector_counts.get(sec, 0) >= max_per_sector:
            continue
        portfolio = pd.concat([portfolio, row.to_frame().T], axis=0)
        used_idx.add(idx)
        sector_counts[sec] = sector_counts.get(sec, 0) + 1

    # Fill if short (relax sector cap)
    if len(portfolio) < n_total:
        for idx, row in ranked.iterrows():
            if len(portfolio) >= n_total:
                break
            if idx in used_idx:
                continue
            portfolio = pd.concat([portfolio, row.to_frame().T], axis=0)
            used_idx.add(idx)

    portfolio = portfolio.sort_values("Final_score", ascending=False).copy()
    portfolio["Weight"] = 1.0 / len(portfolio)

    meta["counts"]["portfolio_size"] = len(portfolio)
    meta["diversification"]["n_sectors_achieved"] = int(portfolio[COLS["sector"]].nunique())

    out_cols = [
        COLS["company"], COLS["name"], COLS["sector"],
        COLS["total_esg"], COLS["E"], COLS["S"], COLS["G"],
        COLS["roe"], COLS["de"],
        "ESG_weighted_risk", "ESG_score",
        "Financial_score", "Final_score",
        "Weight"
    ]
    return portfolio[out_cols].reset_index(drop=True), meta


# ----------------------------
# 7) GUI (structured + summary + charts)
# ----------------------------
esg_cat_dd = widgets.Dropdown(
    options=list(ESG_MAX_BY_CATEGORY.keys()),
    value="Zems (≤20)",
    description="ESG risks:",
    layout=widgets.Layout(width="420px")
)

fin_prof_dd = widgets.Dropdown(
    options=list(FIN_PROFILES.keys()),
    value="Sabalansēts",
    description="Fin. risks:",
    layout=widgets.Layout(width="420px")
)

alpha_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="Ilgtspēja:",
    readout_format=".2f",
    layout=widgets.Layout(width="520px"),
)
alpha_help = widgets.HTML(
    "<div style='color:#555; font-size:12px;'>"
    "0 = tikai finanšu kvalitāte, 1 = tikai ilgtspēja. Vidus = līdzsvars.</div>"
)

beta_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="Atdeve:",
    readout_format=".2f",
    layout=widgets.Layout(width="520px"),
)
beta_help = widgets.HTML(
    "<div style='color:#555; font-size:12px;'>"
    "0 = uzsvars uz stabilitāti (zemāks D/E), 1 = uzsvars uz atdevi (augstāks ROE).</div>"
)

wE_slider = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="E prioritāte:", readout_format=".2f")
wS_slider = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="S prioritāte:", readout_format=".2f")
wG_slider = widgets.FloatSlider(value=0.34, min=0.0, max=1.0, step=0.01, description="G prioritāte:", readout_format=".2f")

n_total_int = widgets.BoundedIntText(value=12, min=3, max=50, step=1, description="N (uzņēmumi):")
n_sectors_int = widgets.BoundedIntText(value=6, min=1, max=20, step=1, description="K (sektori):")
max_per_sector_int = widgets.BoundedIntText(value=3, min=1, max=20, step=1, description="Max / sektors:")

run_btn = widgets.Button(description="Build portfolio", button_style="primary", icon="check")

summary_out = widgets.Output()
result_out = widgets.Output()

def render_summary():
    with summary_out:
        clear_output()

        wE_n, wS_n, wG_n = _normalize_weights(wE_slider.value, wS_slider.value, wG_slider.value)

        esg_max = ESG_MAX_BY_CATEGORY[esg_cat_dd.value]
        prof = FIN_PROFILES[fin_prof_dd.value]
        max_de = prof["max_de"]
        min_roe = prof["min_roe"]

        d = company_100_main_df.copy()
        _require_columns(d)
        for k in ["total_esg","roe","de"]:
            d[COLS[k]] = pd.to_numeric(d[COLS[k]], errors="coerce")
        d = d.dropna(subset=[COLS["total_esg"], COLS["roe"], COLS["de"], COLS["sector"], COLS["company"], COLS["name"]])

        after_filters = d[
            (d[COLS["total_esg"]] <= esg_max) &
            (d[COLS["de"]] <= max_de) &
            (d[COLS["roe"]] >= min_roe)
        ]

        display(widgets.HTML(
            "<div style='padding:10px; border:1px solid #ddd; border-radius:10px;'>"
            "<div style='font-weight:600; margin-bottom:6px;'>Konfigurācijas kopsavilkums</div>"
            f"<div><b>Filtri:</b> Total_ESG ≤ {esg_max} | DebtToEquity ≤ {max_de} | ReturnOnEquity ≥ {min_roe}</div>"
            f"<div><b>E/S/G svari (normalizēti):</b> E={wE_n:.2f}, S={wS_n:.2f}, G={wG_n:.2f}</div>"
            f"<div><b>Līdzsvars:</b> Ilgtspēja={pct_label(alpha_slider.value)} | Finanses={pct_label(1-alpha_slider.value)}"
            f" | Atdeve={pct_label(beta_slider.value)} | Stabilitāte={pct_label(1-beta_slider.value)}</div>"
            f"<div><b>Pēc filtriem paliek:</b> {len(after_filters)} uzņēmumi (no {len(d)})</div>"
            "</div>"
        ))

def on_any_change(change):
    render_summary()

for w in [esg_cat_dd, fin_prof_dd, wE_slider, wS_slider, wG_slider, alpha_slider, beta_slider]:
    w.observe(on_any_change, names="value")

def on_run_clicked(_):
    with result_out:
        clear_output()

        port, meta = build_portfolio(
            df=company_100_main_df,
            esg_category=esg_cat_dd.value,
            fin_profile=fin_prof_dd.value,
            wE=wE_slider.value, wS=wS_slider.value, wG=wG_slider.value,
            alpha_esg_vs_fin=alpha_slider.value,
            beta_roe_vs_de=beta_slider.value,
            n_total=int(n_total_int.value),
            n_sectors=int(n_sectors_int.value),
            max_per_sector=int(max_per_sector_int.value),
        )

        if port is None:
            print(meta["message"])
            return

        # Pretty table
        display(
            port.style.format({
                COLS["total_esg"]:"{:.2f}",
                COLS["E"]:"{:.2f}", COLS["S"]:"{:.2f}", COLS["G"]:"{:.2f}",
                COLS["roe"]:"{:.3f}",
                COLS["de"]:"{:.2f}",
                "ESG_weighted_risk":"{:.3f}",
                "ESG_score":"{:.3f}",
                "Financial_score":"{:.3f}",
                "Final_score":"{:.3f}",
                "Weight":"{:.3f}",
            })
        )

        # Charts
        plot_portfolio_charts(port)

run_btn.on_click(on_run_clicked)

filters_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>1) Riska tolerance (filtri)</h3>"),
    widgets.HBox([esg_cat_dd, fin_prof_dd]),
])

priorities_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>2) Prioritātes (līdzsvars)</h3>"),
    alpha_slider, alpha_help,
    beta_slider, beta_help,
])

esg_weights_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>3) ESG fokuss (E/S/G)</h3>"),
    wE_slider, wS_slider, wG_slider,
    widgets.HTML("<div style='color:#555; font-size:12px;'>Svari tiek automātiski normalizēti, lai summa būtu 1.</div>")
])

portfolio_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>4) Portfeļa uzbūve</h3>"),
    widgets.HBox([n_total_int, n_sectors_int, max_per_sector_int]),
])

ui = widgets.VBox([
    filters_box,
    priorities_box,
    esg_weights_box,
    portfolio_box,
    summary_out,
    run_btn,
    result_out
])

render_summary()
display(ui)